# TCIA COVID-19 预处理（Colab）

严格按照 `preprocess/proc_spatial_sequence.py` 的原始处理逻辑：
- 每个 nii volume：**50 个 npy**（z≥128 时）或 **33 个 npy**（z<128 时）
- patch 大小 128³，ScaleIntensityRangePercentiles(1, 99.9)

In [ ]:
# Cell 1: 挂载 Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2: 安装依赖
!pip install -q nibabel monai

In [ ]:
# Cell 3: TCIA COVID-19 预处理（完整逻辑）
import os
import random
import numpy as np
import nibabel as nib
from monai.transforms import Compose, Resize, ScaleIntensityRangePercentiles
from tqdm import tqdm

# ========== 配置：修改为你的 Google Drive 路径 ==========
# TCIA 数据放在该目录下，支持递归查找 .nii.gz / .nii
# 推荐结构: DRIVE_TCIA_BASE/TCIA_Covid19/CTImagesInCOVID19/volume-xxx.nii.gz
DRIVE_TCIA_BASE = '/content/drive/MyDrive/dataset/TCIA_Covid19'  # 修改为你的路径
DRIVE_SAVE_ROOT = '/content/drive/MyDrive/dataset/TCIA_Covid19/patch_random_spatial'

# ========== 与 proc_spatial_sequence.py 完全一致 ==========
PATCH_NUM = 50
PATCH_SIZE_LIST = [(128, 128, 128)]
TAR_IMG_SIZE = [128, 128, 128]
START_INDEX = 0


def cut_patch(img_array, patch_size):
    img_shape = img_array.shape
    size_z, size_y, size_x = patch_size
    center_x_min, center_x_max = size_x // 2, img_shape[2] - size_x // 2 - 1
    center_y_min, center_y_max = size_y // 2, img_shape[1] - size_y // 2 - 1
    center_z_min, center_z_max = size_z // 2, img_shape[0] - size_z // 2 - 1
    if center_x_min >= center_x_max:
        x1, x2 = 0, img_shape[2]
    else:
        center_x = random.randint(center_x_min, center_x_max)
        x1, x2 = center_x - size_x // 2, center_x + size_x // 2
    if center_y_min >= center_y_max:
        y1, y2 = 0, img_shape[1]
    else:
        center_y = random.randint(center_y_min, center_y_max)
        y1, y2 = center_y - size_y // 2, center_y + size_y // 2
    if center_z_min >= center_z_max:
        z1, z2 = 0, img_shape[0]
    else:
        center_z = random.randint(center_z_min, center_z_max)
        z1, z2 = center_z - size_z // 2, center_z + size_z // 2
    img_patch = img_array[z1:z2, y1:y2, x1:x2]
    return img_patch, [z1, z2, y1, y2, x1, x2]


def load_nii_data(nii_file):
    nii_data = nib.load(nii_file)
    return nii_data.get_fdata(), nii_data.affine


def load_and_patch_transforms(img, tar_img_size):
    transforms = Compose([
        Resize(spatial_size=(tar_img_size[0], tar_img_size[1], tar_img_size[2]), mode='trilinear'),
        ScaleIntensityRangePercentiles(lower=1., upper=99.9, b_min=0.0, b_max=1.0, clip=True, relative=False, channel_wise=False),
    ])
    return transforms(img)


def cut_and_save_one_volume(image_file, patch_size_list, patch_num, save_root, start_index, tar_img_size):
    image, affine = load_nii_data(image_file)
    path_parts = image_file.replace('\\', '/').split('/')
    ds_name = path_parts[-3] if len(path_parts) >= 3 else 'TCIA_Covid19'
    nii_id = path_parts[-1].split('.nii.gz')[0].split('.nii')[0].split('.mha')[0]

    if len(image.shape) == 4:
        return []
    images = [image]
    n, patch_path_list = 0, []

    for image_index, image in enumerate(images):
        image = image.transpose((2, 1, 0))
        _patch_num = int(patch_num / 1.5) if image.shape[0] < patch_size_list[0][2] else patch_num
        for i in range(_patch_num):
            n += 1
            patch_size = random.choice(patch_size_list)
            image_patch, _ = cut_patch(image, patch_size)
            image_patch = image_patch.transpose((2, 1, 0))
            image_patch = load_and_patch_transforms(np.expand_dims(image_patch, 0), tar_img_size).numpy()[0, ...]
            save_name = os.path.join(save_root, '%s_%s_%s_%d.nii.gz' % (ds_name, nii_id, image_index, start_index + n))
            patch_path_list.append(save_name)
            np.save(save_name.replace('.nii.gz', '.npy'), image_patch)
    return patch_path_list


def collect_nii_files(base_path):
    nii_files = []
    for root, _, files in os.walk(base_path):
        for f in files:
            if f.endswith('.nii.gz') or f.endswith('.nii'):
                nii_files.append(os.path.join(root, f))
    return sorted(nii_files)


os.makedirs(DRIVE_SAVE_ROOT, exist_ok=True)
data_list = collect_nii_files(DRIVE_TCIA_BASE)
print(f'找到 {len(data_list)} 个 nii 文件')

if len(data_list) == 0:
    print(f'未找到文件，请检查: {DRIVE_TCIA_BASE}')
else:
    patch_list_all = []
    for i, path in enumerate(tqdm(data_list, desc='预处理')):
        patch_list = cut_and_save_one_volume(path, PATCH_SIZE_LIST, PATCH_NUM, DRIVE_SAVE_ROOT, START_INDEX, TAR_IMG_SIZE)
        patch_list_all += patch_list
        if (i + 1) % 20 == 0:
            print(f'[{i+1}/{len(data_list)}] {path} -> {len(patch_list)} npy')
    print(f'\n完成! 共 {len(patch_list_all)} 个 npy, 输出: {DRIVE_SAVE_ROOT}')

In [ ]:
# Cell 4 (可选): 生成 train_patch_spatial.txt 用于 pretrain
list_save_dir = os.path.join(os.path.dirname(DRIVE_SAVE_ROOT), 'pretrain_lists')
os.makedirs(list_save_dir, exist_ok=True)
npy_files = sorted([os.path.join(DRIVE_SAVE_ROOT, f) for f in os.listdir(DRIVE_SAVE_ROOT) if f.endswith('.npy')])
with open(os.path.join(list_save_dir, 'train_patch_spatial.txt'), 'w') as f:
    f.write('\n'.join(npy_files))
print(f'train_patch_spatial.txt 已保存: {list_save_dir}')